# 真实结果阅读与复核
默认不运行虚构回测。完成真实数据pipeline后，把RUN指向对应目录；没有数据时本notebook只提示缺失，不生成收益。

In [ ]:
from pathlib import Path
import json
import pandas as pd
from IPython.display import display
ROOT = Path.cwd() if (Path.cwd()/'pyproject.toml').exists() else Path.cwd().parent
RUN = ROOT/'results'/'real_run_01'
ready = (RUN/'manifest.json').exists()
print('已找到研究结果' if ready else '尚无完整真实研究结果，请先按README准备数据。')

In [ ]:
if ready:
    manifest = json.loads((RUN/'manifest.json').read_text(encoding='utf-8'))
    print(manifest['status'], manifest['data_kind'], manifest['execution'])
    ready = manifest['status'] == 'complete'

## 清洗覆盖率与隐含carry
先查坏数据，再看收益。某些日期剔除率过高时，评估选择偏差。

In [ ]:
if ready:
    iv = pd.read_csv(RUN/'iv_panel.csv')
    forwards = pd.read_csv(RUN/'forwards.csv')
    display(iv.groupby('date').size().describe())
    display(forwards[['q','parity_pairs','parity_dispersion']].describe())

## 核对账本
逐日权益变化必须等于期权腿+对冲腿-成本。默认初始资本从manifest读取。

In [ ]:
if ready:
    ledger = pd.read_csv(RUN/'ledger_oos.csv')
    capital = manifest['config']['capital']
    change = ledger.equity.diff().fillna(ledger.equity.iloc[0]-capital)
    print('最大对账误差', (change-ledger.net_pnl).abs().max())
    display(ledger.head())
    display(pd.read_csv(RUN/'robustness_oos.csv'))

练习：解释哪种成本最重要；最大亏损日是Gamma路径、方向残差还是交易费用？本工程三腿归因不是完整Greeks高阶归因，不要过度解读。